# Local library call demo (`piboufilings`)

This notebook runs against the local source tree (no package install required).

Default storage backend is DuckDB; install it once with `pip install duckdb`
or pass `export_format="csv"` to `get_filings(...)` for the CSV fallback.


In [1]:
import warnings 

#warnings.filterwarnings('ignore')

In [2]:
from pathlib import Path
import sys

# Ensure local repo root is importable
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Using repo root: {repo_root}')

Using repo root: /Users/pibou/Documents/GitHub/pibou-filings


In [3]:
from piboufilings import get_parser_for_form_type_internal

# Local call on the library (no SEC network request)
parser_13f = get_parser_for_form_type_internal('13F-HR', './demo_local_output')
parser_nport = get_parser_for_form_type_internal('NPORT-P', './demo_local_output')
parser_sec16 = get_parser_for_form_type_internal('SECTION-6', './demo_local_output')

print('13F parser:', type(parser_13f).__name__)
print('NPORT parser:', type(parser_nport).__name__)
print('Section16 parser:', type(parser_sec16).__name__)

13F parser: Form13FParser
NPORT parser: FormNPORTParser
Section16 parser: FormSection16Parser


## Optional: live SEC call using local library
Set `RUN_NETWORK = True` and a real contact email to execute.

In [ ]:
RUN_NETWORK = True

if RUN_NETWORK:
    from piboufilings import get_filings

    get_filings(
        user_name="Your Name",
        user_agent_email="your@email.com",
        cik=None,
        form_type=None,
        start_year=2024,
        end_year=2024,
        base_dir="./demo_local_output",
        log_dir="./demo_local_logs",
        raw_data_dir="./demo_local_raw",
        max_workers=10,
        keep_raw_files=True,
        export_format="duckdb",   # or "csv" if you do not want the duckdb dependency
    )
    print("Download/parse completed.")
else:
    print("Network call skipped. Set RUN_NETWORK=True to execute.")


In [ ]:
# Peek at the DuckDB output written above.
#
# Note: if you re-run the previous cell (`get_filings`), make sure this cell's
# `con` from a prior run is not still open — re-running this cell is safe;
# restarting the kernel is the simplest reset.
import duckdb
from pathlib import Path

db_path = Path("./demo_local_output/piboufilings.duckdb")
if db_path.exists():
    con = duckdb.connect(str(db_path))
    try:
        tables = [row[0] for row in con.execute(
            "SELECT table_name FROM information_schema.tables ORDER BY table_name"
        ).fetchall()]
        print("Tables:", tables)
        for t in tables:
            n = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
            print(f"  {t}: {n:,} rows")
    finally:
        con.close()
else:
    print(f"No DuckDB file at {db_path}; rerun the cell above with RUN_NETWORK=True.")
